In [10]:
# used libraries in our project
import pandas as pd
import re
from scrapers import read_links_from_file, scrape_metadata, write_metadata_to_csv
from classes import AccidentData, MetaData
import spacy
from spacy.matcher import Matcher

In [11]:
# loading spaCy model
nlp = spacy.load('en_core_web_sm')
matcher = Matcher(nlp.vocab)

# loading metadata from previously saved file
links_to_scrape = read_links_from_file("test.txt")
meta_data_list = scrape_metadata(links_to_scrape)
write_metadata_to_csv(meta_data_list, "test0")

# defining vehicle type dictionary (all possible vehicles taken from fast-track file)
vehicle_types = [
    "bus", "car", "noah", "human hauler", "trolley", "chander gari", 
    "auto rickshaw", "CNG", "easy-bike", "truck", "garbage truck", 
    "trailer", "motorcycle", "microbus", "scooter", "construction vehicle", 
    "bicycle", "ambulance", "pickup", "lorry", "paddy cutter vehicles", 
    "bulkhead", "crane", "wrecker", "tractor", "cart", "leguna", 
    "nosimon", "three-wheeler", "four-wheeler", "votvoti", "kariman", 
    "mahindra", "van", "rickshaw", "boat", "trawler", "vessel", 
    "launch", "tanker", "oil tanker", "road roller", "power tiller", 
    "excavator", "train", "airplane", "pedestrian"
]

# adding vehicle pattern to matcher
vehicle_pattern = [{"LEMMA": {"IN": vehicle_types}}]
matcher.add("VEHICLE_TYPE", [vehicle_pattern])

# function for cleaning and deduplicating lists
def clean_and_deduplicate(items):
    seen = set()
    cleaned = []
    for item in items:
        singular = item.lower().rstrip('s')
        if singular not in seen:
            seen.add(singular)
            cleaned.append(item)  # just to keep the original form
    return cleaned

# function for removing duplicates and filtering non-Bangladesh locations
def clean_locations(locations):
    cleaned = []
    for loc in locations:
        if loc not in cleaned and loc != 'Saudi Arabia' and loc != 'UAE':
            cleaned.append(loc)
    return cleaned

# function to extract casualties, injured, and reason
def extract_casualties(doc):
    casualties_pattern = [
        {"LIKE_NUM": True},  
        {"POS": "NOUN", "OP": "+"},  
        {"TEXT": {"FUZZY": {"IN": ["killed", "dead", "fatalities", "casualties", "injured", "victims"]}}},
    ]
    matcher.add("CASUALTIES", [casualties_pattern])
    matches = matcher(doc)
    casualties = [doc[start:end].text for match_id, start, end in matches]
    casualties_age = extract_ages(doc)
    return clean_and_deduplicate(casualties), casualties_age

# function to extract ages
def extract_ages(doc):
    age_pattern = [
        {"POS": "PROPN"},  # first name
        {"POS": "PROPN"},  # last name
        {"IS_PUNCT": True, "OP": "?"},  # some optional punctuation (comma)
        {"IS_DIGIT": True}  # age as a digit
    ]
    matcher.add("AGE_INFO", [age_pattern])
    matches = matcher(doc)
    ages = [doc[start:end].text for match_id, start, end in matches]
    return clean_and_deduplicate(ages)

# function to extract injured persons
def extract_injured(doc):
    injured_pattern = [
        {"LIKE_NUM": True},  
        {"POS": "NOUN", "OP": "+"},  
        {"TEXT": {"FUZZY": {"IN": ["injured", "hurt"]}}},
    ]
    matcher.add("INJURED", [injured_pattern])
    matches = matcher(doc)
    injured = [doc[start:end].text for match_id, start, end in matches]
    return clean_and_deduplicate(injured)

# function to extract reason for the accident
def extract_reason(text):
    pattern = r'(due to|because of)\s+(.*?)(\.|,|$)'
    match = re.search(pattern, text)
    return match.group(2) if match else "Unknown"

# function to extract reason for the accident (TODO)
def extract_sequence_of_actions(text):
    return "Sequence of actions placeholder"

# function to extract details
def extract_details(meta_data_list):
    processed_data = []

    for text in meta_data_list:
        doc = nlp(text.raw_text)
        
        # location, date, and time
        locations = [ent.text for ent in doc.ents if ent.label_ == 'GPE']
        locations = clean_locations(locations)
        date_info = [ent.text for ent in doc.ents if ent.label_ == 'DATE' or ent.label_ == 'TIME']
        
        # vehicles
        matches = matcher(doc)
        vehicles = [doc[start:end].text for match_id, start, end in matches]
        vehicles = clean_and_deduplicate(vehicles)

        # casualties and their ages
        casualties, casualties_age = extract_casualties(doc)
        
        # number of injured persons
        injured = extract_injured(doc)
        
        # reason for the accident
        reason = extract_reason(text.raw_text)

        # sequence of actions (placeholder for now)
        actions = extract_sequence_of_actions(text.raw_text)

        # AccidentData object
        accident_data = AccidentData(
            location= locations, 
            date= date_info,  
            vehicles= vehicles, 
            casualties= casualties, 
            casualties_age= casualties_age, 
            injured= injured, 
            accident_reason= reason, 
            action_sequence= actions, 
            link= text.link
        )
        
        processed_data.append(vars(accident_data))
    
    return processed_data

# processing and saving the data in CSV format
def process_and_save(meta_data_list, output_file):
    processed_data = extract_details(meta_data_list)
    df = pd.DataFrame(processed_data)
    df.to_csv(output_file, sep=';', index=False)

process_and_save(meta_data_list, 'processed_events.csv')


Scrapping [https://www.unb.com.bd/category/Bangladesh/man-killed-in-kushtia-road-crash/4366]
Scrapping [https://www.unb.com.bd/category/Bangladesh/truck-ambulance-collision-leaves-one-dead-in-natore/3597]
Scrapping [https://www.unb.com.bd/category/Bangladesh/lyricist-omar-faruk-dies-in-narsingdi-road-crash/104135]
Scrapping [https://www.unb.com.bd/category/Bangladesh/three-of-a-family-killed-in-kurigram-road-crash/24449]
Scrapping [https://www.unb.com.bd/category/Bangladesh/2-killed-in-chattogram-road-crash/19204]
Scrapping [https://www.unb.com.bd/category/Bangladesh/motorcyclist-killed-38-rmg-workers-hurt-in-manikganj-road-crash/2658]
Scrapping [https://www.unb.com.bd/category/Bangladesh/four-die-in-horrific-road-crash-in-gazipur/102533]
Scrapping [https://www.unb.com.bd/category/Bangladesh/25-medical-students-hurt-in-cumilla-road-crash/8067]
Scrapping [https://www.unb.com.bd/category/Bangladesh/2-motorcyclists-killed-in-narsingdi-road-accident/12404]
Scrapping [https://www.unb.com.bd